# N-Player Deep SRQ Solver Ablation on Level-Based Foraging

Compares baseline N-player SRE, DCA-BL only, Spatial Branch-and-Bound only, and efficient warm start on the compact 3-player LBF setup.

In [4]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "lbf_grid":
    REPO_ROOT = ROOT.parents[1]
elif (ROOT / "discrete_action_space" / "lbf_grid").exists():
    REPO_ROOT = ROOT
else:
    REPO_ROOT = ROOT.parents[1]

for path in [REPO_ROOT, REPO_ROOT / "discrete_action_space", REPO_ROOT / "discrete_action_space" / "bimatrix_game"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from discrete_action_space.lbf_grid.deep_srq_lbf import run_lbf_solver_ablation
from bimatrix_game.stats_utils import load_training_stats, save_training_stats

OUTPUT_ROOT = REPO_ROOT / "discrete_action_space" / "lbf_grid" / "ablation_runs" / "nplayer_solver_ablation"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT

PosixPath('/home/wowthecoder/SRE-DQN/discrete_action_space/lbf_grid/ablation_runs/nplayer_solver_ablation')

In [5]:
# Smoke defaults. Increase N_EPISODES for a full run.
N_EPISODES = 500
USE_GPU = True

COMMON_HP = {
    "batch_size": 16,
    "learning_starts": 100,
    "replay_buffer_capacity": 5000,
    "solver_max_iter": 100,
    "solver_tol": 1e-4,
}

VARIANTS = (
    # {"label": "baseline", "solver_name": "baseline_nplayer"},
    # {"label": "path_mcp", "solver_name": "path_mcp_nplayer"},
    # {"label": "smoothing_newton", "solver_name": "smoothing_newton_nplayer"},
    {"label": "dca_bl_only", "solver_name": "dca_bl_nplayer"},
    {"label": "sbb_only", "solver_name": "sbb_nplayer"},
    {"label": "efficient_warm_start", "solver_name": "warm_start_nplayer"},
)

In [6]:
results = run_lbf_solver_ablation(
    variants=VARIANTS,
    n_episodes=N_EPISODES,
    output_root=OUTPUT_ROOT,
    use_gpu=USE_GPU,
    write_plots=True,
    hyperparameter_overrides=COMMON_HP,
)
save_training_stats(OUTPUT_ROOT / "manifest.txt", results)

LBF DeepSRQ | players=3 | solver=dca_bl_nplayer | eps0=0.5 | schedule=linear | seed=2025


lbf:dca_bl_nplayer_eps0.5_linear__dca_bl_only:   2%|▊                                | 12/500 [11:52<8:02:55, 59.38s/it]


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def rolling(values, window=20):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return values
    window = max(1, min(window, values.size))
    return np.convolve(values, np.ones(window) / window, mode="valid")

fig, ax = plt.subplots(figsize=(12, 5))
for label, stats in results.items():
    joint_rewards = np.sum(np.asarray(stats["rewards"], dtype=float), axis=0)
    ax.plot(rolling(joint_rewards), label=label)
ax.set_title("LBF N-player DeepSRQ joint reward")
ax.set_xlabel("Episode")
ax.set_ylabel("Rolling joint reward")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()

In [ ]:
rows = []
for label, stats in results.items():
    timing = stats["timing"]
    joint_rewards = np.sum(np.asarray(stats["rewards"], dtype=float), axis=0)
    rows.append({
        "variant": label,
        "mean_joint_reward": float(np.mean(joint_rewards)),
        "mean_last_20_joint_reward": float(np.mean(joint_rewards[-20:])),
        "wall_clock_seconds": timing["wall_clock_seconds"],
        "sre_solves": timing["sre_solve_time"]["count"],
        "mean_sre_solve_ms": None if timing["sre_solve_time"]["mean_seconds"] is None else 1000 * timing["sre_solve_time"]["mean_seconds"],
    })
rows